Exception Memory Layout
==============================================================================

Python `Exception` are odd in that:

1.  They typically contain just a string message, and
2.  they have neither a `message: str` attribute, a `message(self) -> str`
    method, nor anything similar.

`Exception` contain only an `args: tuple` attribute. The canonical way to get
the message from an exception is to call `str` on it. The base implementation of
`Exception.__str__` depends on inspecting `args`, essentially:

1.  If `args` is empty, return `""` (the empty string).
2.  If `args` contains one element, call `str` on it and return the result.
3.  Otherwise, return `str(args)`.

You might be tempted to think that `args` serves as generic storage for
exception data, with semantics layered on top. You may imagine the power of
being able to tell everything about _any_ exception from its `type` and `args`.
You would imagine custom error implementations like:

In [1]:
import typing


# V1 uses `Exception.args` as its memory layout
class ArgTypeErrorV1(TypeError):
    """
    A {py:exc}`TypeError` that captures the argument name as {py:attr}`symbol`,
    in addition to the {py:attr}`expected` {py:class}`type` and
    {py:attr}`actual` value.
    """

    # Custom constructor presents modern argument names and types
    def __init__(self, symbol: str, expected: type, actual: typing.Any):
        # Values are ordered and sent to `Exception.__init__` to become `args`
        super().__init__(symbol, expected, actual)

    # Getters provide name/type access into backing `args`

    @property
    def symbol(self) -> str:
        return self.args[0]

    @property
    def expected(self) -> type:
        return self.args[1]

    @property
    def actual(self) -> typing.Any:
        return self.args[2]

    # Custom stringification forms the message
    def __str__(self):
        return (
            f"expected `{self.symbol}` to be a {self.expected!r}, "
            f"given {self.actual!r}"
        )


def print_arg_type_err(err):
    print("type()       ", type(err))
    print("str()        ", err)
    print()
    print(".args        ", err.args)
    print()
    print(".symbol      ", err.symbol)
    print(".expected    ", err.expected)
    print(".actual      ", err.actual)


err_v1 = ArgTypeErrorV1(symbol="x", expected=str, actual=123)
print_arg_type_err(err_v1)

type()        <class '__main__.ArgTypeErrorV1'>
str()         expected `x` to be a <class 'str'>, given 123

.args         ('x', <class 'str'>, 123)

.symbol       x
.expected     <class 'str'>
.actual       123


**However, you would be _wrong_.**

Maybe that was the idea at some point in the distant past, but the present
situation is a total hodgepodge, with the current direction towards storing just
a string message in `args`, with the remaining information in attributes. Hence
you _may_ be able to tell something about an unfamiliar `Exception` subclass by
looking at its `args`, but you can't expect to, and everything else is a
crapshoot.

For an example of some built-in silliness we can take a look at
[`FileNotFoundError`][], a nominal extension of [`OSError`][]:

[`FileNotFoundError`]: https://docs.python.org/3/library/exceptions.html#FileNotFoundError
[`OSError`]: https://docs.python.org/3/library/exceptions.html#OSError

In [2]:
from splatlog.lib import err_catch, check

err = check(err_catch(open, "missing.txt"), FileNotFoundError)


def print_fnf_err(err: FileNotFoundError):
    print("type()       ", type(err))
    print("str()        ", str(err))
    print()
    print(".args        ", err.args)
    print()
    print(".errno       ", err.errno)
    print(".strerror    ", err.strerror)
    print(".filename    ", err.filename)


print_fnf_err(err)

type()        <class 'FileNotFoundError'>
str()         [Errno 2] No such file or directory: 'missing.txt'

.args         (2, 'No such file or directory')

.errno        2
.strerror     No such file or directory
.filename     missing.txt


Notice that _some_ of the exception data is in `args` — `errno` and `strerror`
are, but the `filename` is not. As expected, the concoction is historical:

>  For backwards compatibility, if three arguments are passed, the args
>  attribute contains only a 2-tuple of the first two constructor arguments.
>
> _Source: https://docs.python.org/3/library/exceptions.html#OSError_

Furthermore, the `errno` and `strerror` attributes use separate memory; they do
_not_ reference into `args`:

In [3]:
import errno


err = check(err_catch(open, "missing.txt"), FileNotFoundError)
err.args = (errno.EIO, "I/O screwiness")

print_fnf_err(err)

type()        <class 'FileNotFoundError'>
str()         [Errno 2] No such file or directory: 'missing.txt'

.args         (5, 'I/O screwiness')

.errno        2
.strerror     No such file or directory
.filename     missing.txt


In an attempt to provide something other than indictment, as best as I can synthesize common/encouraged practice implies custom error implementations like:

In [4]:
class ArgTypeErrorV2(TypeError):
    def __init__(self, symbol: str, expected: type, actual: typing.Any):
        # Message is formed immediately and stored in `args[0]`, producing the
        # expected result from the base `__str__` implementation
        super().__init__(
            f"expected `{symbol}` to be a {expected!r}, given {actual!r}"
        )
        self.symbol = symbol
        self.expected = expected
        self.actual = actual


err_v2 = ArgTypeErrorV2(symbol="x", expected=str, actual=123)
print_arg_type_err(err_v2)


type()        <class '__main__.ArgTypeErrorV2'>
str()         expected `x` to be a <class 'str'>, given 123

.args         ("expected `x` to be a <class 'str'>, given 123",)

.symbol       x
.expected     <class 'str'>
.actual       123


However, I think I prefer the following, which abstains from using `args` at all
and produces the message when asked to present itself as a string:

In [5]:
class ArgTypeErrorV3(TypeError):
    def __init__(self, symbol: str, expected: type, actual: typing.Any):
        super().__init__()
        self.symbol = symbol
        self.expected = expected
        self.actual = actual

    def __str__(self):
        return (
            f"expected `{self.symbol}` to be a {self.expected!r}, "
            f"given {self.actual!r}"
        )


err_v3 = ArgTypeErrorV3(symbol="x", expected=str, actual=123)
print_arg_type_err(err_v3)


type()        <class '__main__.ArgTypeErrorV3'>
str()         expected `x` to be a <class 'str'>, given 123

.args         ()

.symbol       x
.expected     <class 'str'>
.actual       123


There are drawbacks:

1.  More verbose.
2.  Sure to cause problems with _something_ that assumes `args[0]` will be
    populated and meaningful.
3.  If the exception is being transported to another system with a separate
    codebase then `__str__` has to be added to the serialization or the string
    templating logic has to be reimplemented at the destination.
4.  If any of the data members are mutated between construction and rendering
    the message it may get very confusing. This is a core issue with any
    structured errors or logs in naturally mutable languages though, so while
    rendering the message immediately may help it would not solve the problem.

However, I feel like it has advantages as well:

1.  **Simpler to understand** — the exception is a data structure. The memory
    layout is exactly that data. You don't need detailed knowledge about how 
    `Exception.args` works. What `str(err)` does is immediately obvious.
2.  **Easier to extend** — a `__str__` message is one way to present the data;
    other presentations such as `__rich__` for [`rich`][] terminal text or
    `__md__` for markdown would fit in naturally.

[`rich`]: https://rich.readthedocs.io/en/latest/



